# 📊 Mutual Fund Analysis Dashboard — Python Pipeline

**Data Source:** [mfapi.in](https://www.mfapi.in) — Free Indian Mutual Fund API  
**Goal:** Identify Top 30 Low-Risk, High-Return Mutual Fund Schemes from 2500+ schemes  
**Tools:** Python · Pandas · Sklearn · Matplotlib · Seaborn · OpenPyXL

---

### Pipeline Overview
1. Load raw dataset (fetched via `generate_dataset.py`)
2. Data Cleaning
3. Descriptive Statistics
4. Normalization (MinMaxScaler)
5. Custom Scoring & Ranking
6. Extract Top 30 Funds
7. Export to Excel
8. Visualizations

In [ ]:
# Install dependencies if needed
# !pip install -r requirements.txt

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✅')

## Step 0 — Fetch Raw Data from mfapi.in

Run the cell below once to download ~2500 mutual fund schemes. Takes ~10–15 minutes.  
**Skip this if `data/mutual_funds_raw.csv` already exists.**

In [ ]:
import subprocess, sys
# Uncomment to fetch data:
# subprocess.run([sys.executable, 'generate_dataset.py'], check=True)

## Step 1 — Load Dataset

In [ ]:
RAW_CSV  = 'data/mutual_funds_raw.csv'
TOP30_XLSX = 'data/top_30_mutual_funds.xlsx'
CHARTS_DIR = 'charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

df = pd.read_csv(RAW_CSV)
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
df.head()

## Step 2 — Data Cleaning

In [ ]:
initial = len(df)

# Remove blank scheme names
df = df[df['scheme_name'].notna() & (df['scheme_name'] != '')]

# Convert numeric columns
for col in ['latest_nav', 'return_1y', 'return_3y', 'fund_age_years']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing key metrics
df = df.dropna(subset=['return_1y', 'return_3y', 'latest_nav'])

# Remove invalid NAV
df = df[df['latest_nav'] > 0]

# Remove extreme return outliers
df = df[(df['return_1y'] >= -100) & (df['return_1y'] <= 500)]
df = df[(df['return_3y'] >= -100) & (df['return_3y'] <= 500)]

# Standardize text columns
for col in ['scheme_type', 'scheme_category', 'scheme_sub_category', 'fund_house']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown').str.strip().str.title()

df = df.reset_index(drop=True)
print(f'Rows before: {initial}  |  After: {len(df)}  |  Removed: {initial - len(df)}')
df.info()

## Step 3 — Descriptive Statistics

In [ ]:
df[['latest_nav', 'return_1y', 'return_3y', 'fund_age_years']].describe()

In [ ]:
print('Fund distribution by Scheme Type:')
print(df['scheme_type'].value_counts().to_string())
print()
print('Top 10 Categories:')
print(df['scheme_category'].value_counts().head(10).to_string())

### Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['return_1y'].dropna(), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['return_1y'].mean(), color='red', linestyle='--', label=f"Mean: {df['return_1y'].mean():.1f}%")
axes[0].set_title('Distribution of 1-Year Returns', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Return (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(df['return_3y'].dropna(), bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(df['return_3y'].mean(), color='red', linestyle='--', label=f"Mean: {df['return_3y'].mean():.1f}%")
axes[1].set_title('Distribution of 3-Year Returns', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/01_return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Normalization

In [ ]:
scaler = MinMaxScaler()
features = ['return_1y', 'return_3y', 'fund_age_years', 'latest_nav']

df_norm = df.copy()
df_norm[features] = scaler.fit_transform(df[features])

print('Normalized — sample values (all in [0,1]):')
df_norm[features].describe()

## Step 5 — Custom Scoring & Ranking

**Scoring Formula:**
| Feature | Weight | Reason |
|---|---|---|
| 3-Year Return | 40% | Long-term performance |
| 1-Year Return | 30% | Recent momentum |
| Fund Age | 15% | Stability indicator |
| Latest NAV | 15% | Established fund signal |

Only funds with **1-Year Return > 0** are eligible.

In [ ]:
# Filter: positive 1Y return only
eligible_mask = df['return_1y'] > 0
eligible_norm = df_norm[eligible_mask].copy()
eligible_orig = df[eligible_mask].copy()

print(f'Eligible funds (1Y return > 0): {eligible_mask.sum()} out of {len(df)}')

# Apply scoring formula
eligible_norm['score'] = (
    0.40 * eligible_norm['return_3y'] +
    0.30 * eligible_norm['return_1y'] +
    0.15 * eligible_norm['fund_age_years'] +
    0.15 * eligible_norm['latest_nav']
)

eligible_orig['score'] = eligible_norm['score'].values
ranked = eligible_orig.sort_values('score', ascending=False).reset_index(drop=True)

print(f'Top scorer: {ranked.iloc[0]["scheme_name"]}')
print(f'Score: {ranked.iloc[0]["score"]:.4f}')

## Step 6 — Top 30 Mutual Funds

In [ ]:
top30 = ranked.head(30).copy()
top30.index = range(1, 31)

display_cols = ['scheme_name', 'fund_house', 'scheme_category', 'return_1y', 'return_3y', 'fund_age_years', 'score']
top30[display_cols].style.format({'return_1y': '{:.2f}%', 'return_3y': '{:.2f}%', 
                                   'fund_age_years': '{:.1f} yrs', 'score': '{:.4f}'})

## Step 7 — Export to Excel

In [ ]:
os.makedirs('data', exist_ok=True)
with pd.ExcelWriter(TOP30_XLSX, engine='openpyxl') as writer:
    top30.to_excel(writer, sheet_name='Top 30 Funds', index=True)
    ranked.to_excel(writer, sheet_name='All Ranked Funds', index=False)

print(f'✅ Saved to {TOP30_XLSX}')

## Step 8 — Visualizations

In [ ]:
# Chart: Fund count by scheme type
fig, ax = plt.subplots(figsize=(9, 5))
type_counts = df['scheme_type'].value_counts().head(8)
bars = ax.barh(type_counts.index, type_counts.values, color=sns.color_palette('muted', len(type_counts)))
ax.bar_label(bars, padding=3)
ax.set_title('Number of Funds by Scheme Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/02_funds_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Avg 3Y Return by Category
fig, ax = plt.subplots(figsize=(12, 6))
cat_returns = df.groupby('scheme_category')['return_3y'].mean().dropna().sort_values(ascending=False).head(10)
bars = ax.bar(range(len(cat_returns)), cat_returns.values, color=sns.color_palette('viridis', len(cat_returns)))
ax.set_xticks(range(len(cat_returns)))
ax.set_xticklabels(cat_returns.index, rotation=30, ha='right', fontsize=9)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_title('Average 3-Year Return by Fund Category (Top 10)', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg 3-Year Return (%)')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/03_avg_3y_return_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Top 10 AMCs by avg 1Y return
fig, ax = plt.subplots(figsize=(12, 6))
amc_returns = (
    df.groupby('fund_house')['return_1y']
    .agg(['mean','count'])
    .query('count >= 5')
    .sort_values('mean', ascending=False)
    .head(10)
)
bars = ax.barh(amc_returns.index, amc_returns['mean'], color='#2196F3')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Top 10 AMCs by Average 1-Year Return (min 5 schemes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg 1-Year Return (%)')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/04_top_amc_by_return.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Top 30 composite scores
fig, ax = plt.subplots(figsize=(12, 10))
labels = [f"{i+1}. {n[:45]}" for i, n in enumerate(top30['scheme_name'])]
bars = ax.barh(labels[::-1], top30['score'].values[::-1], color='#FF9800')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
ax.set_title('Top 30 Mutual Funds — Composite Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Composite Score')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/05_top30_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Top 30 — 1Y vs 3Y Return scatter
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(top30['return_1y'], top30['return_3y'],
                c=top30['score'], cmap='YlOrRd', s=120, edgecolors='gray', linewidth=0.5)
plt.colorbar(sc, ax=ax, label='Composite Score')
for _, row in top30.head(10).iterrows():
    ax.annotate(row['scheme_name'][:25], (row['return_1y'], row['return_3y']), fontsize=7, alpha=0.8)
ax.set_title('Top 30 Funds — 1-Year vs 3-Year Returns', fontsize=13, fontweight='bold')
ax.set_xlabel('1-Year Return (%)')
ax.set_ylabel('3-Year Return (%)')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/06_top30_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Pie — Top 30 by Category
fig, ax = plt.subplots(figsize=(8, 8))
cat_dist = top30['scheme_category'].value_counts()
ax.pie(cat_dist.values, labels=cat_dist.index, autopct='%1.1f%%',
       colors=sns.color_palette('pastel', len(cat_dist)), startangle=140, pctdistance=0.8)
ax.set_title('Top 30 Funds — Category Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/07_top30_category_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart: Correlation Heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[['latest_nav', 'return_1y', 'return_3y', 'fund_age_years']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Key Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/08_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ All charts saved to charts/')

## 🔍 Key Insights

| Insight | Detail |
|---|---|
| 🏆 Top Fund | Based on scoring formula above |
| 📈 Best 3Y Category | Equity funds typically lead |
| 💰 Avg SIP Min | Varies by fund |
| 📉 Low Risk picks | High score + moderate age |

---

**Next Step:** Open `data/top_30_mutual_funds.xlsx` to explore the filtered results,  
or load `data/mutual_funds_raw.csv` into **Power BI** for an interactive dashboard.